# 라이브 댓글 관리 — 모델 선정 스파이크

**스파이크/평가용 코드입니다. 프로덕션 코드가 아닙니다.**

라이브 방송 채팅에서 욕설/혐오표현을 걸러내기 위한 후보 모델들을 실제로 로드해서 비교합니다.

- 실시간 경량 분류기: [`smilegate-ai/kor_unsmile`](https://huggingface.co/smilegate-ai/kor_unsmile), [`JminJ/kcElectra_base_Bad_Sentence_Classifier`](https://huggingface.co/JminJ/kcElectra_base_Bad_Sentence_Classifier)
- 비실시간 생성형 LLM: [`kakaocorp/kanana-1.5-2.1b-instruct-2505`](https://huggingface.co/kakaocorp/kanana-1.5-2.1b-instruct-2505) (경량) / `kakaocorp/kanana-1.5-8b-instruct-2505` (고성능, 수동 실행)

모델을 바꾸려면 `model_registry.py`의 `ACTIVE_CLASSIFIER` / `ACTIVE_GENERATIVE` 값만 바꾸고 해당 셀들을 다시 실행하면 됩니다.

In [1]:
%pip install -q -r ../requirements-test.txt

Note: you may need to restart the kernel to use updated packages.


## 1. 실시간 경량 분류기 후보

In [1]:
from loaders import load_sequence_classifier
from model_registry import ACTIVE_CLASSIFIER, CLASSIFIER_MODELS

model_id = CLASSIFIER_MODELS[ACTIVE_CLASSIFIER]
bundle = load_sequence_classifier(model_id)
print(f"loaded: {bundle.name} on device={bundle.device}")
print(f"problem_type: {bundle.model.config.problem_type}")
print(f"labels: {bundle.model.config.id2label}")

c:\Users\SSAFY\workspace\S15P11A108\AI\test\model_selection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\SSAFY\workspace\S15P11A108\AI\test\model_selection\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SSAFY\.cache\huggingface\hub\models--smilegate-ai--kor_unsmile. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode 

loaded: smilegate-ai/kor_unsmile on device=cpu
problem_type: multi_label_classification
labels: {0: '여성/가족', 1: '남성', 2: '성소수자', 3: '인종/국적', 4: '연령', 5: '지역', 6: '종교', 7: '기타 혐오', 8: '악플/욕설', 9: 'clean'}


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [2]:
import torch

samples = [
    "이 옷 진짜 예쁘네요 판매자님 감사합니다",
    "가격 좀 너무 비싼거 아니에요?",
    "옷 상태 사진이랑 다른데 사기 아니에요?",
    "저런 걸 옷이라고 파네 진짜 미친놈인가",
    "여자가 무슨 명품을 입고 방송을 하냐 꼴값이네",
]

with torch.no_grad():
    inputs = bundle.tokenizer(
        samples, padding=True, truncation=True, return_tensors="pt"
    ).to(bundle.device)
    logits = bundle.model(**inputs).logits

id2label = bundle.model.config.id2label
is_multilabel = bundle.model.config.problem_type == "multi_label_classification"
scores = torch.sigmoid(logits) if is_multilabel else torch.softmax(logits, dim=-1)

for text, score_row in zip(samples, scores):
    ranked = sorted(
        ((id2label[i], float(s)) for i, s in enumerate(score_row)),
        key=lambda x: x[1],
        reverse=True,
    )
    top = ranked[:3] if is_multilabel else ranked[:1]
    print(repr(text))
    for label, score in top:
        print(f"    {label}: {score:.3f}")

'이 옷 진짜 예쁘네요 판매자님 감사합니다'
    clean: 0.926
    악플/욕설: 0.048
    여성/가족: 0.022
'가격 좀 너무 비싼거 아니에요?'
    clean: 0.928
    악플/욕설: 0.055
    지역: 0.019
'옷 상태 사진이랑 다른데 사기 아니에요?'
    clean: 0.911
    악플/욕설: 0.072
    지역: 0.017
'저런 걸 옷이라고 파네 진짜 미친놈인가'
    악플/욕설: 0.914
    clean: 0.067
    기타 혐오: 0.024
'여자가 무슨 명품을 입고 방송을 하냐 꼴값이네'
    여성/가족: 0.542
    악플/욕설: 0.487
    clean: 0.079


다른 분류기로 바꾸려면 `model_registry.py`의 `ACTIVE_CLASSIFIER`를 `"kcelectra_bad_sentence"`로 바꾸고 위 두 셀을 다시 실행하면 됩니다.

## 2. 비실시간 경로 — kanana-1.5-instruct zero-shot 프롬프팅

In [3]:
from loaders import load_causal_lm
from model_registry import ACTIVE_GENERATIVE, GENERATIVE_MODELS

gen_model_id = GENERATIVE_MODELS[ACTIVE_GENERATIVE]
gen_bundle = load_causal_lm(gen_model_id)
print(f"loaded: {gen_bundle.name} on device={gen_bundle.device}")

c:\Users\SSAFY\workspace\S15P11A108\AI\test\model_selection\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SSAFY\.cache\huggingface\hub\models--kakaocorp--kanana-1.5-2.1b-instruct-2505. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package 

loaded: kakaocorp/kanana-1.5-2.1b-instruct-2505 on device=cpu


In [5]:
from loaders import generate_chat

PROMPT_TEMPLATE = (
    "다음은 라이브 커머스 방송 채팅 댓글이다. 욕설, 혐오표현, 판매자에 대한 "
    "인신공격이 포함되어 있는지 판단하고 '포함' 또는 '미포함'으로 답한 뒤, "
    "한 줄로 이유를 설명하라.\n\n댓글: {comment}"
)

for text in samples:
    result = generate_chat(gen_bundle, PROMPT_TEMPLATE.format(comment=text))
    print(f"{text!r}\n  -> {result}\n")

'이 옷 진짜 예쁘네요 판매자님 감사합니다'
  -> 포함  
이 댓글은 판매자에 대한 직접적인 비난이나 욕설, 혐오표현이 없지만, 감사의 표현과 함께 옷에 대한 칭찬이 섞여 있어, 판매자에 대한 긍정적인 반응으로 볼 수 있으나, 특별히 인신공격이나 부정적인 언급이 없어 '포함'이라고 판단하기 어렵다. 단순히 감사의 인사로 볼 수 있으나, 다른 댓글에서 욕설, 혐오표현, 인신공격이 있다면 그에 맞춰 판단해야 한다.



'가격 좀 너무 비싼거 아니에요?'
  -> 포함

이 댓글은 판매자의 제품 가격에 대해 단순히 의견을 표현한 것이지만, 비판적인 어투로 가격이 높다고 지적하고 있어 판매자에 대한 불만의 뉘앙스가 내포되어 있으므로 '포함'이 적절하다.



'옷 상태 사진이랑 다른데 사기 아니에요?'
  -> 포함  
이 댓글은 제품의 사진과 실제 상품의 상태가 다르다는 불만을 표현하고 있어, 판매자에 대한 직접적인 비난이나 욕설, 혐오 표현은 없지만, 사기 의혹을 제기하고 있어 판매자에 대한 의문을 제기하는 내용이 포함되어 있습니다.



'저런 걸 옷이라고 파네 진짜 미친놈인가'
  -> 포함  
이 댓글은 판매자가 만든 옷에 대해 부정적인 평가를 하며, '미친놈'이라는 표현으로 인신공격적 요소가 포함되어 있기 때문입니다.



'여자가 무슨 명품을 입고 방송을 하냐 꼴값이네'
  -> 포함  
이 댓글은 여성의 외모나 의상에 대한 비하적 표현을 사용하여, 상대방을 무시하거나 비난하는 혐오 표현의 성격을 띠고 있기 때문입니다.



### 참고 — 8B 모델 / Ollama GGUF 경량 대안 (수동 실행 전용, 자동 검증 대상 아님)

`kanana-1.5-8b-instruct-2505`는 다운로드/RAM 부담이 커서(~16GB+) 이 노트북에서 자동으로 실행하지 않습니다. 직접 시도하려면 `model_registry.py`의 `ACTIVE_GENERATIVE`를 `"kanana-8b"`로 바꾸고 위 두 셀을 다시 실행하세요.

로컬 리소스가 부족하면 커뮤니티 GGUF 양자화본을 Ollama로 pull해서 대신 써볼 수 있습니다 (정확한 quant 태그는 저장소에서 직접 확인할 것):

```
ollama pull hf.co/tensorblock/kakaocorp_kanana-1.5-8b-instruct-2505-GGUF:Q4_K_M
```

In [ ]:
# 선택 사항 — 로컬에 Ollama가 설치되어 있고 위 모델을 pull한 경우에만 실행.
# 이 셀은 스모크 테스트 대상이 아닙니다.
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "hf.co/tensorblock/kakaocorp_kanana-1.5-8b-instruct-2505-GGUF:Q4_K_M",
        "prompt": PROMPT_TEMPLATE.format(comment=samples[0]),
        "stream": False,
    },
    timeout=60,
)
print(response.json()["response"])